# Fall Detection: IMU Classifier Training (v2.0)
This notebook trains a neural network to distinguish between **FALL** and **NORMAL** (Negative) events using 1 second of IMU data (Accel/Gyro).

### 1. Setup and Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

# Parameters from IMU_Fall_Capture.ino
SAMPLES_PER_GESTURE = 120
NUM_CHANNELS = 6 # ax, ay, az, gx, gy, gz
CLASSES = ['fall', 'normal']

### 2. Load and Preprocess Data
This section reads your CSV files and splits them into individual 1-second "gestures."

In [ ]:
def load_gesture_data(filename, class_id):
    # Load raw CSV (ignoring headers if they exist)
    df = pd.read_csv(filename, header=None, comment='-')
    data = df.values
    
    # Split data into 120-sample blocks
    num_gestures = len(data) // SAMPLES_PER_GESTURE
    data = data[:num_gestures * SAMPLES_PER_GESTURE]
    reshaped_data = data.reshape(num_gestures, SAMPLES_PER_GESTURE * NUM_CHANNELS)
    
    labels = np.full((num_gestures,), class_id)
    return reshaped_data, labels

# Load your collected data
try:
    fall_data, fall_labels = load_gesture_data('fall_data.csv', 0)
    normal_data, normal_labels = load_gesture_data('normal_data.csv', 1)

    # Combine and Shuffle
    X = np.vstack((fall_data, normal_data))
    y = np.concatenate((fall_labels, normal_labels))
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.1, random_state=42)

    print(f"Data Loaded Successfully!")
    print(f"Total Samples: {len(X)}")
    print(f"Training: {len(X_train)}, Validation: {len(X_val)}, Test: {len(X_test)}")
except FileNotFoundError:
    print("Error: CSV files not found. Please run your data collector first!")

### 3. Model Architecture
We use a Dense (MLP) network for motion classification, which is lightweight for the Nano 33.

In [ ]:
model = models.Sequential([
    layers.Input(shape=(SAMPLES_PER_GESTURE * NUM_CHANNELS,)),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(16, activation='relu'),
    layers.Dense(len(CLASSES), activation='softmax')
])

model.compile(optimizer='adam', 
              loss='sparse_categorical_crossentropy', 
              metrics=['accuracy'])

history = model.fit(X_train, y_train, 
                    validation_data=(X_val, y_val), 
                    epochs=100, 
                    batch_size=8)

### 4. Evaluating the Model
This section tracks the training progress and helps you detect if the model is overfitting.

In [ ]:
# Plot Training & Validation Accuracy/Loss
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('IMU Model Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('IMU Model Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend()
plt.show()

### 5. Confusion Matrix
This heatmap shows how many 'Normal' movements were incorrectly flagged as 'Falls'.

In [ ]:
y_pred = np.argmax(model.predict(X_test), axis=1)
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=CLASSES, yticklabels=CLASSES, cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Fall Detection Confusion Matrix')
plt.show()

### 6. Export to TensorFlow Lite
Finally, convert the model to a C-header file for deployment on the Arduino Nano 33 BLE Sense.

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

# Save the model to disk
with open("fall_model.tflite", "wb") as f:
    f.write(tflite_model)

print("Model converted to TFLite format!")